<a href="https://colab.research.google.com/github/gustkt/GE338-Lab-4/blob/main/lab4_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Geographic Modeling — สร้างและตรวจสอบแบบจำลองเชิงพื้นที่

* **Situation**

**ความเสี่ยงต่อการสะสมของ PM2.5 ในพื้นที่ภาคเหนือ**

* **ภารกิจที่ 1: กำหนดกรอบแนวคิด (Conceptual Framework)**

"พื้นที่ใดในภาคเหนือของประเทศไทยมีความเสี่ยงต่อการสะสม PM2.5 สูง จากปัจจัยด้านแหล่งกำเนิด ภูมิประเทศ และสภาพอุตุนิยมวิทยา?"

จากงานวิชาการที่ได้ค้นคว้า พบว่าปัจจัยที่ส่งผลต่อการสะสมของ PM2.5 ในพื้นที่ภาคเหนือ ได้แก่
 1. Biomass burning ซึ่งเป็นแหล่งกำเนิดหลัก ของ PM2.5 ในพื้นที่ภาคเหนือ (Chotamonsak et al., 2026)
 2. ภูมิประเทศแบบภูเขา-หุบเขา ทำให้เกิดการสะสมของ PM2.5 (Chotamonsak et al., 2026)
 3. Temperature inversion + ลมอ่อน ทำให้ PM2.5 ไม่กระจาย (Inlaung et al.,2024)
 4. ไร่หลังเก็บเกี่ยว (พื้นที่ที่มี NDVI ต่ำ) มักเป็นโอกาสสำหรับการเผา

โดยน้ำหนักของแต่ละปัจจัยมาจาก
AHP โดย AHP Matrix ได้มาจากหลักฐานทางวิชาการที่ค้นคว้า
และใช้ข้อมูลจาก API กรมควบคุมมลพิษ (Air4Thai) ในการ Validate โมเดล

* **ภารกิจที่ 2: สร้างและรัน Spatial Model**

In [ ]:
# ติดตั้ง + import
!pip install -q geemap earthengine-api geemap geopandas mss

In [ ]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import requests
from datetime import datetime

In [ ]:
# Authenticate + Initialize
ee.Authenticate()
ee.Initialize(project='ee-gustkt45513')

In [ ]:
# กำหนดพื้นที่ศึกษา (ภาคเหนือ)
# Northern Thailand (filter จังหวัด)
provinces = ee.FeatureCollection('FAO/GAUL/2015/level1')

roi = provinces.filter(
    ee.Filter.inList('ADM1_NAME', [
        'Chiang Mai','Chiang Rai','Lamphun','Lampang',
        'Phayao','Nan','Phrae','Mae Hong Son','Uttaradit'
    ])
)

Map = geemap.Map()
Map.centerObject(roi, 7)
Map.addLayer(roi, {}, 'ROI')
# Map

In [ ]:
# Fire Hotspot (MODIS FIRMS)
fire = ee.ImageCollection('FIRMS') \
    .filterDate('2025-11-01', '2026-03-30') \
    .select('T21') \
    .mean() \
    .clip(roi)

In [ ]:
# NDVI (Sentinel-2)
# ฟังก์ชัน mask cloud (SCL)
def mask_s2_clouds_scl(image):
    scl = image.select('SCL')

# คลาสที่เราต้องการ "กรองออก" (Mask out)
  # 3: Cloud Shadow
  # 8: Cloud Medium Probability
  # 9: Cloud High Probability
  # 10: Thin Cirrus
  # 11: Snow
    bad_classes = [3, 8, 9, 10]

    mask = scl.remap(
        bad_classes,
        ee.List.repeat(0, len(bad_classes)),
        1
    )

    return image.updateMask(mask).divide(10000)

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate('2025-11-01', '2026-03-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds_scl) \
    .median()

ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI').clip(roi)

In [ ]:
# Elevation + Slope (SRTM)
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)
slope = ee.Terrain.slope(dem)

In [ ]:
# Wind (ERA5)
wind = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
    .filterDate('2025-11-01', '2026-03-30') \
    .select(['u_component_of_wind_10m', 'v_component_of_wind_10m']) \
    .map(lambda img: img.expression(
        "sqrt(u*u + v*v)", {
            'u': img.select('u_component_of_wind_10m'),
            'v': img.select('v_component_of_wind_10m')
        }).rename('wind_speed')
    ) \
    .mean() \
    .clip(roi)

In [ ]:
# Normalize Function (ใช้ Min-Max : X′=(X−min)/(max−min))
def normalize(img):
    stats = img.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=1000,
        maxPixels=1e13
    )
    band_name = ee.String(img.bandNames().get(0))
    min_val = ee.Number(stats.get(band_name.cat('_min')))
    max_val = ee.Number(stats.get(band_name.cat('_max')))
    return img.subtract(min_val).divide(max_val.subtract(min_val))

In [ ]:
# Normalize ทุก factor
fire_n = normalize(fire)
ndvi_n = normalize(ndvi)
elev_n = normalize(dem)
slope_n = normalize(slope)
wind_n = normalize(wind)

# inverse factors เพราะ Elevation ต่ำ → เสี่ยง, Wind ต่ำ → เสี่ยง และ NDVI ต่ำ → เสี่ยง
elev_inv = ee.Image(1).subtract(elev_n)
wind_inv = ee.Image(1).subtract(wind_n)
ndvi_inv = ee.Image(1).subtract(ndvi_n)
slope_inv = ee.Image(1).subtract(slope_n)

| Factor    | ทำ             |
| --------- | -------------- |
| Elevation | 1 - normalized |
| Slope     | 1 - normalized |
| Wind      | 1 - normalized |
| NDVI      | 1 - normalized |

จากการ review งานวิชาการพบว่า
* ไฟ หรือ biomass burning เป็นแหล่ง PM2.5 หลัก คิดเป็นประมาณ 23–38% ของ PM2.5 ทั้งหมด (Thongsame et al., 2025) **จึงให้ Weight สูงสุด**

* ลมมีผลต่อการเคลื่อนที่ และการกระจาย ของ PM2.5 (Kanchanachat et al., 2026)
    
    **Fire vs Wind ให้ Weight 2
    (ทั้งคู่สำคัญ แต่ Fire ยังเป็น source หลัก)
    Wind vs NDVI ให้ Weight 4**

* ภาคเหนือมีภูเขา และหุบเขา (Elevation สูง) ทำให้ PM2.5 สะสม หรือกักเก็บมลพิษ (Inlaung et al.,2024) และ Slope มีผลต่อการไหลของอากาศแต่ เป็นปัจจัยที่สำคัญน้อยกว่า elevation

    **Elevation vs NDVI ให้ Weight 3 Elevation vs Slope ให้ Weight 2**

* ไร่หลังเก็บเกี่ยว (NDVI ต่ำ) มักเป็นโอกาสสำหรับการเผาก็จริง แต่น้ำหนักจะน้อยกว่าปัจจัยอื่นถ้าวิเคราะห์เฉพาะการสะสมของ PM2.5
    
    **NDVI ให้ Weight ต่ำกว่า Fire / Wind / Elevation**

| Factor    | หลักฐาน                  | ผลใน AHP |
| --------- | ------------------------ | -------- |
| Fire      | source หลัก  | สูงสุด   |
| Wind      | คุมการกระจาย           | รอง      |
| Elevation | ตัวดักมลพิษ                 | กลาง     |
| NDVI      | เชื้อเพลิง                     | ต่ำ      |
| Slope     | secondary                | ต่ำสุด   |


ความสัมพันธ์ทั้งหมดจะถูกแปลงเป็นมาตราส่วนของ Saaty เพื่อสร้าง Matrix การเปรียบเทียบแบบ AHP

|       | Fire | Wind | Elev | NDVI | Slope |
| ----- | ---- | ---- | ---- | ---- | ----- |
| Fire  | 1    | 2    | 3    | 5    | 5     |
| Wind  | 1/2  | 1    | 2    | 4    | 3     |
| Elev  | 1/3  | 1/2  | 1    | 3    | 2     |
| NDVI  | 1/5  | 1/4  | 1/3  | 1    | 2     |
| Slope | 1/5  | 1/3  | 1/2  | 1/2  | 1     |


In [ ]:
# AHP PAIRWISE MATRIX
A = np.array([
    [1,   2,   3,   5,   5],
    [1/2, 1,   2,   4,   3],
    [1/3, 1/2, 1,   3,   2],
    [1/5, 1/4, 1/3, 1,   2],
    [1/5, 1/3, 1/2, 1/2, 1]
])

In [ ]:
# คำนวณ Weight (EIGENVECTOR METHOD)

eigvals, eigvecs = np.linalg.eig(A)

# หา eigenvalue ที่มากที่สุด
max_index = np.argmax(eigvals)
max_eigval = np.real(eigvals[max_index])

# eigenvector หลัก
weights = np.real(eigvecs[:, max_index])

# normalize ให้ผลรวม = 1
weights = weights / weights.sum()

print("Weights:")
for i, w in enumerate(weights):
    print(f"Factor {i+1}: {w:.3f}")

In [ ]:
# แสดงผลเป็นชื่อ Factor
factors = ['Fire', 'Wind', 'Elevation', 'NDVI', 'Slope']

for f, w in zip(factors, weights):
    print(f"{f}: {w:.3f}")

In [ ]:
# CONSISTENCY CHECK [Consistency Ratio (CR)]

n = A.shape[0]

# Consistency Index (CI)
CI = (max_eigval - n) / (n - 1)

# Random Index (RI) table
RI_dict = {
    1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90,
    5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41,
    9: 1.45, 10: 1.49
}

RI = RI_dict[n]

CR = CI / RI

print("\nConsistency Check")
print(f"λ_max = {max_eigval:.3f}")
print(f"CI = {CI:.3f}")
print(f"CR = {CR:.3f}")

# ตีความผล
if CR < 0.1:
    print("✅ Matrix is consistent (CR < 0.1)")
else:
    print("❌ Matrix is NOT consistent (CR >= 0.1)")

In [ ]:
# เอาไปใช้ใน GEE
w_fire, w_wind, w_elev, w_ndvi, w_slope = weights

In [ ]:
# combine model
risk = (
    fire_n.multiply(w_fire)
    .add(wind_inv.multiply(w_wind))
    .add(elev_inv.multiply(w_elev))
    .add(ndvi_n.multiply(w_ndvi))
    .add(slope_n.multiply(w_slope))
).rename('PM25_Risk')

In [ ]:
stats = risk.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=1000,
    maxPixels=1e13
)

print(stats.getInfo())

In [ ]:
# เพื่อให้ map มีค่า 0–1 จริง และ compare ง่าย จึงต้อง Normalize risk อีกครั้ง

min_val = 0.2347120955007702
max_val = 0.7227235309643171

risk_norm = risk.subtract(min_val).divide(max_val - min_val)

In [ ]:
# SMOOTHING data ด้วย focal mean เนื่องจาก hotspots ข้อมูลเป็นจุด ๆ ไม่ได้เต็มพื้นที่

risk_smooth = risk_norm.focal_mean(radius=3000, units='meters')

In [ ]:
# คำนวณ percentileที่ 25,50,75 เพื่อใช้ในการ Classification
percentiles = risk_norm.reduceRegion(
    reducer=ee.Reducer.percentile([25,50,75]),
    geometry=roi,
    scale=1000,
    maxPixels=1e13
)

print(percentiles.getInfo())

In [ ]:
# สร้าง Map

# BASEMAP

Map.add_basemap('HYBRID')

# Define visualization parameters for the risk layer
vis = {
    'min': 0,
    'max': 1,
    'palette': [
        '#001a00', '#003300', '#004d00', '#006600', '#008000', # Dark Green to Green
        '#99cc00', '#ccff00', # Yellow Green to Bright Yellow
        '#ffcc00', '#ff9900', # Orange Yellow to Orange
        '#ff6600', '#ff3300', '#ff0000', # Orange Red to Red
        '#b30000', '#800000' # Dark Red to Maroon
    ]
}

# RISK MAP
Map.addLayer(risk_smooth, vis, 'PM2.5 Risk (Smoothed)')


# CLASSIFICATION ใช้ Threshold ตาม percentile ที่ 25,50,75
risk_class = risk_smooth.expression(
    "(b < 0.23069146825211947) ? 1"
    ": (b < 0.3164525699541888) ? 2"
    ": (b < 0.3946456481437706) ? 3"
    ": 4",
    {'b': risk_smooth}
).toInt().rename('Risk_Class')

risk_class = risk_class.updateMask(risk_norm.mask())

Map.addLayer(risk_class, {
    'min': 1,
    'max': 4,
    'palette': ['green','yellow','orange','red']
}, 'Risk Class')


# ROI
Map.addLayer(
    roi.style(color='white', fillColor='00000000', width=3),
    {},
    'Study Area'
)


# LEGEND
legend_dict = {
    'Low Risk': (0, 128, 0),       # Green
    'Moderate Risk': (255, 255, 0),  # Yellow
    'High Risk': (255, 165, 0),    # Orange
    'Very High Risk': (255, 0, 0)    # Red
}

Map.add_legend(
    title='PM2.5 Risk Level',
    legend_dict=legend_dict,
    position='bottomright'
)


# LABEL จังหวัด
provinces = ee.FeatureCollection('FAO/GAUL/2015/level1').filterBounds(roi)

labels = provinces.map(lambda f: ee.Feature(
    f.geometry().centroid(),
    {'label': f.get('ADM1_NAME')}
))

Map.add_labels(
    data=labels,
    column='label',
    font_size='10pt',
    font_color='white'
)

Map

จากแผนที่ความเสี่ยงการสะสม PM2.5 พบความเสี่ยงการสะสม PM2.5 ระดับสูง-สูงมาก กระจายทั่วภาคเหนือ

* **ภารกิจที่ 3: Sensitivity Analysis**

จาก AHP Fire/Biomass Burning เป็นปัจจัยที่สำคัญที่สุด (0.426)

In [ ]:
# ปรับ weight ±20%

# SENSITIVITY (+20%)

w_fire_up = w_fire * 1.2
w_wind_dn = w_wind * 0.8  # ปรับสมดุล

risk_up = (
    fire_n.multiply(w_fire_up)
    .add(wind_inv.multiply(w_wind_dn))
    .add(elev_inv.multiply(w_elev))
    .add(ndvi_n.multiply(w_ndvi))
    .add(slope_inv.multiply(w_slope))
).rename('PM25_Risk')

# SENSITIVITY (-20%)
w_fire_dn = w_fire * 0.8
w_wind_up = w_wind * 1.2

risk_dn = (
    fire_n.multiply(w_fire_dn)
    .add(wind_inv.multiply(w_wind_up))
    .add(elev_inv.multiply(w_elev))
    .add(ndvi_n.multiply(w_ndvi))
    .add(slope_inv.multiply(w_slope))
).rename('PM25_Risk')

In [ ]:
# คำนวณความแตกต่างในรูปแบบ Earth Engine Image เพื่อแสดงผลบนแผนที่
diff_up_ee = risk_up.subtract(risk)
diff_dn_ee = risk_dn.subtract(risk)

# ดูความแตกต่าง (Uncertainty)
uncertainty_ee = diff_up_ee.abs().add(diff_dn_ee.abs())

Map.addLayer(uncertainty_ee, {
    'min': 0,
    'max': 0.3,
    'palette': ['green','yellow','red']  # green = robust, yellow = moderate, red = sensitive
}, 'Uncertainty Map (EE)')
Map

In [ ]:
# สร้าง Mask สำหรับพื้นที่ที่มีความเปลี่ยนแปลงอย่างมีนัยสำคัญ (Uncertainty > 0.1)
significant_change = uncertainty_ee.gt(0.1)

# แสดงผลเฉพาะพื้นที่ที่ Sensitive ด้วยสีแดง
Map.addLayer(significant_change.updateMask(significant_change), {
    'palette': ['red']
}, 'Sensitive Areas')
Map.to_image(filename='/content/Sensitive Areas.png')
Map

จากภาพ พบพื้นที่ Sensitive บริเวณรอยต่อทางตะวันตกเฉียงใต้ระหว่างจ.แม่ฮ่องสอนกับจ.เชียงใหม่

In [ ]:
# เปรียบเทียบโมเดลทั้ง 3 ตัว ว่าโมเดลไวต่อการเปลี่ยนน้ำหนักแค่ไหน

# ดึงข้อมูลทั้ง 3 ตัว
df_base = geemap.ee_to_df(
    risk.sample(region=roi, scale=1000, numPixels=5000)
)

df_up = geemap.ee_to_df(
    risk_up.sample(region=roi, scale=1000, numPixels=5000)
)

df_dn = geemap.ee_to_df(
    risk_dn.sample(region=roi, scale=1000, numPixels=5000)
)

# ทำให้ DataFrames มีขนาดเท่ากันสำหรับเปรียบเทียบ
min_len = min(len(df_base), len(df_up), len(df_dn))
df_base = df_base.head(min_len)
df_up = df_up.head(min_len)
df_dn = df_dn.head(min_len)

In [ ]:
# เช็คชื่อ column ว่าตรงกันหรือไม่
print(df_base.columns)
print(df_up.columns)
print(df_dn.columns)

In [ ]:
# กราฟ Scatter Plot เพื่อเปรียบเทียบความเสี่ยงพื้นฐาน (Base Risk) กับความเสี่ยงที่ได้จากการเพิ่มน้ำหนักปัจจัย 'Fire/Biomass Burning' ขึ้น 20%
plt.scatter(df_base['PM25_Risk'], df_up['PM25_Risk'], alpha=0.3)
plt.xlabel('Base Risk')
plt.ylabel('Risk (+20% Fire)')
plt.title('Sensitivity (+20%)')

# เพิ่มเส้นแนวทแยง y=x
min_risk_val = min(df_base['PM25_Risk'].min(), df_up['PM25_Risk'].min())
max_risk_val = max(df_base['PM25_Risk'].max(), df_up['PM25_Risk'].max())
plt.plot([min_risk_val, max_risk_val], [min_risk_val, max_risk_val], color='red', linestyle='--')
# plt.savefig('Sensitivity (+20%).png')
plt.show()

In [ ]:
# กราฟ Scatter Plot เพื่อเปรียบเทียบความเสี่ยงพื้นฐาน (Base Risk) กับความเสี่ยงที่ได้จากการลดน้ำหนักปัจจัย 'Fire/Biomass Burning' ลง 20%
plt.scatter(df_base['PM25_Risk'], df_dn['PM25_Risk'], alpha=0.3)
plt.xlabel('Base Risk')
plt.ylabel('Risk (-20% Fire)')
plt.title('Sensitivity (-20%)')

# เพิ่มเส้นแนวทแยง y=x
min_risk_val = min(df_base['PM25_Risk'].min(), df_dn['PM25_Risk'].min())
max_risk_val = max(df_base['PM25_Risk'].max(), df_dn['PM25_Risk'].max())
plt.plot([min_risk_val, max_risk_val], [min_risk_val, max_risk_val], color='red', linestyle='--')
# plt.savefig('Sensitivity (-20%).png')
plt.show()

จากกราฟ Scatter Plot ทั้งสองภาพแสดงให้เห็นว่าจุดส่วนใหญ่กระจุกตัวอยู่ใกล้เส้นประ x=y เป็นรูปแบบ linear ชัดเจน ซึ่งแสดงให้เห็นว่าโมเดลมีความเสถียรสูง (robust) แต่ก็ยังมีจุดที่หลุดออกจาก linear แสดงถึงความไม่แน่นอน (Sensitive) ในบางพื้นที่

In [ ]:
diff_up = np.abs(df_up['PM25_Risk'] - df_base['PM25_Risk'])
diff_dn = np.abs(df_dn['PM25_Risk'] - df_base['PM25_Risk'])

print("Mean diff (+20%):", diff_up.mean())
print("Mean diff (-20%):", diff_dn.mean())

และเมื่อพิจารณาจากค่า Mean ที่ต่างกัน แสดงให้เห็นการตอบสนองของโมเดลที่ ไม่เท่ากันต่อการเพิ่ม/ลด weight โมเดล +20% diff ต่ำมาก ในขณะที่โมเดล -20% diff สูงกว่า (0.684) ชี้ให้เห็นว่าโมเดลพึ่งพา fire factor สูง

* **ภารกิจที่ 4: Validation กับข้อมูลจริง**

In [ ]:
# Use URL for Air4Thai latest data
url = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"

try:
    response = requests.get(url)
    data = response.json()

    records = []

    # List of northern provinces to filter
    northern_provinces = [
        'เชียงใหม่', 'เชียงราย', 'ลำพูน', 'ลำปาง',
        'พะเยา', 'น่าน', 'แพร่', 'แม่ฮ่องสอน', 'อุตรดิตถ์'
    ]

    for station in data['stations']:
        try:
            # Check if station is in Northern Thailand
            area = station.get('areaTH', '')
            if not any(p in area for p in northern_provinces):
                continue

            # Extract PM2.5 value and coordinates
            pm25 = float(station['AQILast']['PM25']['value'])
            lat = float(station['lat'])
            lon = float(station['long'])
            station_id = station['stationID']
            name = station['nameTH']

            records.append({
                'stationID': station_id,
                'nameTH': name,
                'lat': lat,
                'long': lon,
                'PM25': pm25,
                'datetime': datetime.now()
            })
        except (KeyError, TypeError, ValueError):
            continue

    # Create DataFrame outside the loop
    df = pd.DataFrame(records)
    if not df.empty:
        display(df.head())
        print(f"Successfully fetched {len(df)} stations from Northern Thailand.")
    else:
        print("No northern stations found with valid PM2.5 data.")

except Exception as e:
    print(f"Error connecting to Air4Thai: {e}")

In [ ]:
# แปลงเป็น GEE point
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.long, df.lat),
    crs="EPSG:4326"
)

fc = geemap.geopandas_to_ee(gdf)

In [ ]:
# sample ค่า model
samples = risk.sampleRegions(
    collection=fc,
    properties=['PM25'],
    scale=1000
)

sample_df = geemap.ee_to_df(samples)

In [ ]:
# Correlation

r = np.corrcoef(sample_df['PM25_Risk'], sample_df['PM25'])[0,1]
print("Correlation:", r)

จากค่า Correlation ≈ 0.5717 แสดงให้เห็นความสัมพันธ์ระดับปานกลางค่อนไปทางดี พื้นที่ที่โมเดลให้ risk สูง มีแนวโน้ม PM2.5 สูงจริง กล่าวคือโมเดลสามารถอธิบาย spatial pattern ของ PM2.5 ได้ค่อนข้างดี